# 18 pamoka: AI agentų apsauga naudojant kriptografinius kvitus

## Praktinis užrašų knygelės pavyzdys

Ši užrašų knygelė pateikia keturis pratimus:

1. **Pasirašykite pirmąjį kvitą** agento įrankio kvietimui ir patikrinkite jį.
2. **Pakeiskite kvitą** ir stebėkite, kaip nepavyksta patikra.
3. **Sukurkite trijų kvitų grandinę** ir patvirtinkite grandinės vientisumą.
4. **Apvyniokite Microsoft Agent Framework įrankio kvietimą**, kad kiekvienas veiksmas sugeneruotų kvitą.

Visi kriptografiniai primityvai importuojami iš gerai prižiūrimų bibliotekų (`pynacl` Ed25519, `jcs` RFC 8785 kanoninėms JSON reikšmėms, `hashlib` iš Python standartinės bibliotekos SHA-256). Pats kvitų logikos kodas yra paprastas Python, kurį galite perskaityti ir modifikuoti.

Vykdykite langelius iš eilės. Kiekviena dalis yra trumpa ir savarankiška.


## Diegimas

Įdiekite dvi priklausomybes. Abi turi leidžiančias licencijas (Apache-2.0 / MIT).


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## Pagalbinės priemonės

Šie du pagalbiniai įrankiai tvarko base64url kodavimą (be užpildo) ir SHA-256 maišos sukurimą iš bet kokių objektų. Jie leidžia likusiai užrašų knygelės daliai susitelkti ties kvito logika.


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Section 1: Pasirašykite savo pirmąjį kvitą

Įsivaizduokite, kad mūsų agentas **Contoso Travel** ką tik paieškojo skrydžių iš Sidnėjaus į Los Andželą klientui. Norime įrašyti šį įrankio kvietimą kaip pasirašytą kvitą, kad būsimas auditas galėtų jį patikrinti be būtinybės mums pasitikėti.

### 1.1 veiksmas: Sugeneruokite pasirašymo raktą

Produkcijoje agente pasirašymo raktas būtų saugomas aparatūros saugumo modulyje (HSM), Azure Key Vault arba panašioje saugioje saugykloje. Šiai pamokai mes generuojame naują raktą atmintyje.


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### 1.2 žingsnis: Sukurti kvito naštą

Našta apima viską, ką norime, kad kvitas patvirtintų: kas veikė, kokį įrankį, su kokiais argumentais, kas grįžo, pagal kokią politiką ir kada. Argumentus ir rezultatą maišome, o ne įtraukiame tiesiogiai į tekstą, kad kvitas neatiduotų jautrios informacijos.


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### 1.3 žingsnis: Pasirašykite ir surinkite kvitą

Trys veiksmai:

1. Canonicalizuokite turinį naudodami JCS, kad dvi įgyvendinimo versijos, kurios sukuria tą patį loginį kvitą, pagamintų bitų požiūriu identiškus duomenis.
2. Apdorokite canonicalizuotus baitus su SHA-256 žymekliu.
3. Pasirašykite hešą naudodami Ed25519 privatų raktą.

Parašas tuomet pridedamas prie pradinio turinio, kad būtų gautas galutinis kvitas.


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### 1.4 žingsnis: Patvirtinkite kvitą

Patvirtinimas vyksta atlikus procesą atvirkščiai. Pašaliname parašą, iš naujo apskaičiuojame kanoninį maišos kodą ir patikriname parašą pagal kvite esančią viešąją raktą.

Auditorius, atliekantis šį patvirtinimą, nieko papildomo iš mūsų nereikalauja, išskyrus patį kvitą. Nereikia kviesti jokios paslaugos, ieškoti rakto kataloge ar pasitikėti kažkuo.


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

Turėtumėte matyti `Receipt is valid: True`. Agentas sukūrė pirmąjį kriptografiškai pasirašytą audito įrašą.


## 2 skyrius: Pakeisti kvitą

Visa kvitų esmė yra ta, kad juos sunku klastoti nepastebėjus. Įsitikinkime tuo.

Mes pakeisime tik vieną kvito simbolį ir stebėsime, kaip nepavyks patvirtinti.


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### Kas ką tik įvyko?

Kai pakeitėme `policy_id`, pasikeitė kanoniniai baitai. Tų baitų SHA-256 maišos funkcija pasikeitė. Parašas (kuris buvo ant pradinio maišo) nebeatitinka naujo maišo. Patikra teisingai grąžina `False`.

Nėra būdo pakeisti bet kurį kvito lauką ir vis tiek turėti galiojančią patikrą, nebent užpuolikas turi privatų raktą. Kol privatus raktas yra saugykloje, o viešasis raktas paskelbtas, klastojimo slėpti neįmanoma.

Pabandykite patys: pakeiskite `tool_name` arba `agent_id` arba `timestamp` aukščiau esančioje ląstelėje ir paleiskite iš naujo. Kiekvienas pakeitimas sukuria negaliojantį kvitą.


## 3 skyrius: Grandinėnu sujungti kvitus

Vienas kvitas apsaugo vieną veiksmą. Dauguma agentų atlieka daugybę veiksmų. Norint, kad visa seka būtų apsaugota nuo klastojimo, kiekvienas kvitas yra susietas su ankstesniu, į naujo kvito naudingojo krovinio dalį įtraukiant ankstesnio kvito hešą.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

Jei kas nors pašalina ar pertvarko kvitą, grandinėlė nutrūksta būtent tuo tašku. Vėlesnio kvito patikrinimas nepavyksta, nes jo `previous_receipt_hash` nebeatitinka faktinio ankstesnio kvito hešo.


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Dabar nutraukite grandinę, klastodami vidurinį kvitą, ir patikrinkite dar kartą. Klastotas kvitas nepraeina parašo patikrinimo, O KITAS kvitas nepraeina grandinės jungties patikrinimo (nes jo `previous_receipt_hash` nebeatsako pakeisto vidurinio kvito maišos).


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Kvitas 0 vis dar patvirtinamas (jis nebuvo modifikuotas ir neturi ankstesnio elemento, nuo kurio priklausytų). Kvitas 1 nepraeina parašo patikros, nes mes pakeitėme `tool_args_hash`. Kvitas 2 nepraeina grandinės nuorodos patikros, nes jo `previous_receipt_hash` buvo apskaičiuotas remiantis originaliu (dabar modifikuotu) kvitu 1.

Net jei užpuolikas perpasirašytų modifikuotą kvitą 1 (ką jie negali padaryti be privataus rakto), grandinės nuorodos neatitikimas kvite 2 vis tiek atskleistų klastojimą. Norėdamas paslėpti pakeitimą, užpuolikas turėtų perpasirašyti kiekvieną kvitą nuo pakeitimo taško, o tam reikalingas privatusis raktas.


## 4 skyrius: Agentų įrankio kvietimo apsukimas su kvito pasirašymu

Realiame diegime nenorite, kad kiekvienas agento autorius prisimintų iškviesti `make_receipt`. Norite, kad kvito pasirašymas būtų automatinis kiekvienam įrankio kvietimui.

Čia yra paprasčiausias modelis: apvyniotojo klasė, kuri priima bet kokią iškviečiamą įrankio funkciją ir grąžina jos versiją, kuri generuoja kvitus. Tai prisitaiko prie bet kokios agento sistemos, įskaitant Microsoft Agent Framework (`agent_framework.azure`).

Jei neturite sukonfigūruoto Azure AI Foundry projekto, žemiau pateiktas vietinis modelis vis tiek demonstruoja šį modelį.


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to AzureAIProjectAgentProvider as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")

### Integracija su Microsoft Agent Framework

Aukščiau pateiktas `ReceiptedTool` įvyniojimas yra nepriklausomas nuo framework'o. Norėdami jį naudoti agente, sukurtoje naudojant Microsoft Agent Framework, užregistruokite įvyniotą funkciją kaip įrankį. Pažymi, kaip tai atrodytų (vietoje maketo naudotumėte tikrą Azure AI Foundry įrankio registraciją):

```python
# Pseudokodas, rodantis integracijos formą.
# iš agent_framework.azure importuoti AzureAIProjectAgentProvider
# iš azure.identity importuoti AzureCliCredential
#
# provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())
# agent = provider.create_agent(
#     instructions="Jūs esate Contoso Travel agentas ..."
#     tools=[receipted_lookup],   # supakuotas įrankis, o ne žali funkcija
# )
# response = agent.run("Raskite skrydžius iš Sidnėjaus į Los Andželą birželį.")
#
# # Po vykdymo, kiekvienas agento kvietimas į įrankį turi pasirašytą kvitą:
# audit_chain = receipted_lookup.receipts
```

Agentų framework'ui nereikia nieko žinoti apie kvitus. Kvito pasirašymas yra įvyniotas į įrankį, o ne įkištas į framework'ą. Taip pridedate kilmės įrašus prie esamo agento kodo be agento perrašymo.


## Apžvalga ir iššūkis pratęsti

Jūs:

- Sugeneravote Ed25519 raktų porą.
- Sukūrėte ir pasirašėte kvitą agento įrankio kvietimui.
- Patikrinote kvitą neprisijungę, naudodami tik viešąjį raktą.
- Pakeitėte kvitą ir stebėjote patikros nesėkmę.
- Sukūrėte trijų kvitų grandinę su maišos grandinėlėmis.
- Pakeitėte grandinės vidurį ir stebėjote tiek parašo, tiek grandinės nuorodos gedimą.
- Apgaubėte įrankio funkciją automatinio kvitų pasirašymo mechanizmu.

**Iššūkis pratęsti.** Praplėskite kvito schemą su lauku `request_id` (UUID, skirtas paskirstytai sekai). Atitinkamai atnaujinkite `make_receipt`, kad įtrauktų šį lauką, ir patvirtinkite, jog kvitai vis dar pilnai patikrinami. Tada pakeiskite lauką po pasirašymo ir patvirtinkite, kad patikra nepavyksta. Tai privers jus suvokti, kaip kiekvienas kanoninio kodavimo baitas prisideda prie parašo.

**Svarbi riba.** Kvitai įrodo tris dalykus ir tik tris: priskyrimą (ši raktas pasirašė šį turinį), vientisumą (turinys nuo pasirašymo nebuvo pakeistas) ir tvarką (šis kvitas buvo po to kvito). Jie NEįrodo, kad agento veiksmas buvo teisingas, kad `policy_id` nurodyta politika iš tiesų buvo įvertinta arba kad agentas laikėsi kiekvienos taisyklės. Kvitai yra pamatą. Valdymas yra sistema, kurią statote virš jų.

Dar kartą perskaitykite pamokos README, turėdami omenyje šią ribą. Dažniausia klaida, kurią daro komandos su kvitais, yra manyti, kad „mes turime kvitus“ reiškia „mes esame valdomi“. Nėra. Kvitai leidžia patikrinti agento elgesį. Jie nepadaro jo teisingu.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Atsakomybės apribojimas**:
Šis dokumentas buvo išverstas naudojant dirbtinio intelekto vertimo paslaugą [Co-op Translator](https://github.com/Azure/co-op-translator). Nors siekiame tikslumo, prašome atkreipti dėmesį, kad automatiniai vertimai gali turėti klaidų ar netikslumų. Originalus dokumentas jo gimtąja kalba laikomas autoritetingu šaltiniu. Svarbiai informacijai rekomenduojama naudoti profesionalų žmogiškąjį vertimą. Mes neatsakome už jokius nesusipratimus ar neteisingą interpretaciją, kilusią naudojantis šiuo vertimu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
